In [1]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
config_path = os.environ.get("DATUM_API_CONFIG_PATH", "../ops/datum_api_config.json")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [2]:
# Import basic modules
import pandas as pd
from datum_api_client import DatumApi
import datetime
from datetime import timedelta
from typing import Optional, List, Dict, Any


# Import warnings
import warnings
warnings.filterwarnings("ignore")
# pip install xlrd
# pip install openpyxl

In [3]:
from __future__ import annotations

# ── PairFlux stage 1: shuffle final.parquet into one file per benchmark ───────────────────
#
# Why a shuffle at all: final.parquet is sorted by ticker, but PairFlux needs every ticker of
# one benchmark ALIGNED ON THE SAME TIMESTAMPS. Streaming ticker-by-ticker (the OpenDoor /
# DayTwo pattern) cannot do that, and loading the whole file to pivot it is not an option at
# this universe size. So: one sequential pass writes a small per-benchmark parquet holding
# only [ticker, sdate, smin, stack], already cropped to the three class windows. Stage 2 then
# reads one benchmark at a time and pivots it, which is what makes the memory bounded.
#
# Re-run stage 2 with different thresholds as often as you like — the shuffle is the slow
# part and only has to be redone when the source data or the class windows change.

CLASS_WINDOWS_DEFAULT = {
    "PRE":   ((21, 0), (9, 30)),   # crosses midnight
    "OPEN":  ((9, 0), (10, 0)),    # deliberately overlaps the tail of PRE
    "INTRA": ((10, 0), (16, 0)),
}


def _to_smin(hm, session_split_min):
    """Session minutes. Rows at/after session_split_min belong to the NEXT session day, so
    they are numbered NEGATIVE (21:00 -> -180) and the whole 21:00 -> 16:00 span becomes one
    monotonically increasing axis. Without this the PRE window would wrap around midnight and
    every overnight episode would be cut in half."""
    t = hm[0] * 60 + hm[1]
    return t - 24 * 60 if t >= session_split_min else t


def pairflux_stage1_shuffle(
    input_path: str,
    stage_dir: str,
    *,
    class_windows: dict = None,
    session_split_min: int = 1020,        # 17:00
    start_date: Optional[str] = None,     # "YYYY-MM-DD", session date, inclusive
    bench_whitelist: Optional[List[str]] = None,
    STOCK_NUM_FIELD: str = "Stack%",
    log_every_n_chunks: int = 20,
):
    import gc, time, shutil
    import numpy as np
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq
    from pathlib import Path

    if class_windows is None:
        class_windows = CLASS_WINDOWS_DEFAULT

    bounds = [(_to_smin(a, session_split_min), _to_smin(b, session_split_min))
              for a, b in class_windows.values()]
    smin_lo = min(lo for lo, _ in bounds)
    smin_hi = max(hi for _, hi in bounds)

    start_i = int(start_date.replace("-", "")) if start_date else -1

    stage = Path(stage_dir)
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True, exist_ok=True)

    schema = pa.schema([
        ("ticker", pa.string()),
        ("sdate", pa.int32()),
        ("smin", pa.int16()),
        ("stack", pa.float32()),
    ])
    writers = {}
    counts = {}

    def _writer(bench):
        if bench not in writers:
            safe = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in str(bench))
            writers[bench] = pq.ParquetWriter(str(stage / f"{safe}.parquet"), schema,
                                              compression="zstd")
            counts[bench] = 0
        return writers[bench]

    t0 = time.time()
    total_in = total_out = 0
    pf = pq.ParquetFile(input_path)
    wanted = ["ticker", "dt", "bench", STOCK_NUM_FIELD]
    cols = [c for c in wanted if c in pf.schema.names]
    missing = set(wanted) - set(cols)
    if missing:
        raise KeyError(f"final.parquet is missing required columns: {sorted(missing)}")

    print(f"START PairFlux stage1  file={input_path}")
    print(f"  session_split={session_split_min}min  smin window=[{smin_lo}, {smin_hi}]  start_date={start_date}")

    try:
        for ci in range(pf.num_row_groups):
            df = pf.read_row_group(ci, columns=cols).to_pandas()
            total_in += len(df)

            dt = pd.to_datetime(df["dt"], errors="coerce", utc=True)
            ok = dt.notna().to_numpy(copy=False)
            if not ok.any():
                continue
            dt = dt[ok]
            df = df.loc[ok]

            t_arr = (dt.dt.hour.to_numpy(dtype="int32", copy=False) * 60 +
                     dt.dt.minute.to_numpy(dtype="int32", copy=False))
            late = t_arr >= session_split_min
            smin = np.where(late, t_arr - 24 * 60, t_arr).astype("int16")
            # a row after the split belongs to TOMORROW's session
            sess = dt + pd.to_timedelta(np.where(late, 1, 0), unit="D")
            sdate = (sess.dt.year.to_numpy(dtype="int32", copy=False) * 10000 +
                     sess.dt.month.to_numpy(dtype="int32", copy=False) * 100 +
                     sess.dt.day.to_numpy(dtype="int32", copy=False)).astype("int32")

            stack = pd.to_numeric(df[STOCK_NUM_FIELD], errors="coerce").to_numpy(dtype="float32", copy=False)

            keep = (smin >= smin_lo) & (smin <= smin_hi) & np.isfinite(stack)
            if start_i > 0:
                keep &= sdate >= start_i
            if not keep.any():
                continue

            out = pd.DataFrame({
                "ticker": df["ticker"].to_numpy(copy=False)[keep].astype(str),
                "sdate": sdate[keep],
                "smin": smin[keep],
                "stack": stack[keep],
                "bench": df["bench"].to_numpy(copy=False)[keep],
            })
            out = out[pd.notna(out["bench"])]
            out["bench"] = out["bench"].astype(str).str.strip().str.upper()
            out = out[out["bench"] != ""]
            if bench_whitelist:
                wl = {str(b).strip().upper() for b in bench_whitelist}
                out = out[out["bench"].isin(wl)]
            if out.empty:
                continue

            for bench, part in out.groupby("bench", sort=False):
                tbl = pa.Table.from_pandas(part[["ticker", "sdate", "smin", "stack"]],
                                           schema=schema, preserve_index=False)
                _writer(bench).write_table(tbl)
                counts[bench] += len(part)
                total_out += len(part)

            del df, out
            if (ci + 1) % log_every_n_chunks == 0:
                el = time.time() - t0
                print(f"[rg {ci+1:>4}/{pf.num_row_groups}] in={total_in:,} staged={total_out:,} "
                      f"benches={len(writers)} elapsed={el:.1f}s")
                gc.collect()
    finally:
        for w in writers.values():
            w.close()

    print(f"DONE stage1 in={total_in:,} staged={total_out:,} elapsed={time.time()-t0:.1f}s")
    for b, n in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"  {b:<10} rows={n:,}")
    return {b: str(stage / f"{b}.parquet") for b in counts}

In [4]:
# ── PairFlux stage 2: per-benchmark pair scan ─────────────────────────────────────────────


def pairflux_stats_exporter(
    stage_dir: str,
    *,
    output_onefile_jsonl: str = "PAIRFLUX/onefile.jsonl",
    output_summary_csv: str = "PAIRFLUX/summary.csv",
    output_best_pairs_jsonl: str = "PAIRFLUX/best_pairs.jsonl",
    # one line per divergence episode — the only file that can answer "what happened on
    # 2026-07-14 for this pair"; summary/onefile carry all-history aggregates only.
    output_episodes_jsonl: str = "PAIRFLUX/episodes.jsonl",
    write_episodes: bool = True,
    class_windows: dict = None,
    # ONSET windows: a class may only COUNT divergences that were born inside this narrower
    # slice, while still using the full class window to look for the convergence. OPEN is
    # the motivating case: "did the deviations that appeared between 9:00 and 9:25 normalise
    # by 10:00" — a divergence starting at 9:45 is a different question and must not be
    # mixed into the same rate. Classes absent from this dict use their full window.
    onset_windows: dict = None,          # {"OPEN": ((9, 0), (9, 25))}
    # An episode already diverged on the FIRST candle of its session day cannot be dated:
    # it may have been running since the overnight session and only looks like it started
    # at the window open. True drops those; set False to count them as onsets anyway.
    require_fresh_onset: bool = True,
    session_split_min: int = 1020,
    # candle size; None = infer from the staged data (mode of the positive smin steps)
    bar_minutes: Optional[int] = None,
    # "ols"  -> dev = Stack%_A - (alpha + beta*Stack%_B), beta/alpha fitted per (pair, class)
    # "unit" -> dev = Stack%_A - Stack%_B, the plain "both should have moved the same %"
    hedge_mode: str = "ols",
    # episode thresholds, in z units of the pair's own spread (scale-free across pairs)
    div_z: Optional[float] = 2.0,
    conv_z: Optional[float] = 0.5,
    # Absolute thresholds in PERCENTAGE POINTS, ANDed with the z ones. z alone answers "is
    # this unusual for this pair", which is not the same question as "is this worth trading":
    # on a tight pair like AAAU/GLD a clean z=2.4 divergence measures 0.06pp. Set a side to
    # None to drop that condition; at least one divergence condition must remain.
    div_abs_pp: Optional[float] = None,
    conv_abs_pp: Optional[float] = None,
    # How the z scale is estimated. "std" is the textbook z-score, but it has a trap: a pair
    # that spends a large slice of the window diverged inflates its own sigma, so the very
    # divergence you are hunting stops clearing div_z and the pair silently scores 0 episodes.
    # "mad" (median / 1.4826*MAD) takes the scale from the QUIET state instead, so long or
    # frequent divergences stay visible. Try "mad" first if a class comes back suspiciously empty.
    scale_mode: str = "std",
    # Where "dev == 0" sits. Mean/OLS centring puts zero at the pair's AVERAGE spread, which
    # drifts off the resting state whenever divergences are one-sided — and then an absolute
    # conv_abs_pp band around zero is unreachable no matter how the pair behaves. Median
    # centring puts zero at the state the pair actually spends most of its time in, which is
    # what an absolute threshold needs. "auto" = median as soon as anything depends on the
    # resting state (any *_abs_pp threshold, or scale_mode="mad").
    center_mode: str = "auto",       # "zero" | "mean" | "median" | "auto"
    # Economic floor: drop episodes whose peak deviation is below this many percentage points.
    # A spread can be statistically extreme and still be too small to trade.
    min_abs_peak_pp: float = 0.0,
    # Ceiling on the peak. A 60pp gap between two stocks' daily moves is single-name news or
    # a stale print, not a spread that was ever going to close — and it drags SIG up while
    # pushing RATE down. 0 = no ceiling.
    max_abs_peak_pp: float = 0.0,
    # NORMALISATION: both the divergence peak and the return-to-zero must survive this many
    # CONSECUTIVE candles. Single-candle spikes and single-candle touches of zero are noise
    # and must not create or resolve an episode.
    min_hold: int = 3,
    # "Consecutive" candles are decided on the CLOCK, not on row adjacency. Overnight and
    # pre-market bars are irregular (measured on real data: ~3 bars per ticker per overnight
    # session, median step 4 min), so demanding three strictly 1-minute-apart candles makes
    # an episode almost impossible to form there. A gap wider than this many minutes breaks
    # the run; None = require the exact inferred bar step (strict).
    max_gap_minutes: Optional[int] = None,
    # candidate filter (step 1 of the classic pair-trading checklist)
    min_corr: float = 0.7,
    # "Moves synchronously" means beta near 1. A 3x leveraged ETF against its own index is
    # geared, not synchronous: its spread is a mechanical function of the underlying move,
    # not a mispricing that has to revert. beta_band=1.5 keeps only pairs with beta inside
    # [1/1.5, 1.5]; None = no filter. Measured on the first pp-threshold run: 56% of the
    # top-200 INTRA pairs were geared-ETF relationships.
    beta_band: Optional[float] = None,
    # Correlation is measured on k-bar returns, not 1-bar. One-minute returns are mostly
    # microstructure noise, so 1-bar correlation between two ordinary stocks sits around
    # 0.2-0.4 and the 0.7-0.8 rule of thumb (which comes from DAILY data) would reject
    # everything. 5-bar returns are far more stable. If a class prints "no pair reaches
    # corr>=...", the log also prints the best corr actually seen — tune against that.
    corr_step_bars: int = 5,
    max_pairs_per_bench: int = 20000,
    corr_max_rows: int = 20000,          # subsample rows for the corr matmuls only
    # coverage guards
    min_bars_per_ticker: int = 500,
    min_days_per_ticker: int = 10,
    max_tickers_per_bench: int = 800,
    max_matrix_mb: int = 2000,
    # output filter
    min_total: int = 5,                  # keep a pair if ANY class reaches this many episodes
    # Evidence bar for the RANKED list specifically. score = rate_lb * sig lets a large sig
    # buy back a weak rate_lb, so a pair with 5 episodes and a 4pp spread can top the table
    # on almost no evidence. None = same as min_total.
    best_min_total: Optional[int] = None,
    top_k_best: int = 500,
    # Augmented Dickey-Fuller on the spread. Off by default: it costs far more than every
    # other statistic combined and, because Stack% resets to 0 every session, the pooled
    # series it runs on is a concatenation of daily segments rather than one long process.
    # half_life / mr_lambda below are day-aware and answer the same practical question.
    compute_adf: bool = False,
    adf_maxlag: int = 1,
    log_every_n_pairs: int = 5000,
):
    """
    PairFlux: rate how reliably a pair of same-benchmark tickers CONVERGES after diverging.

    Deviation (the thing that diverges):
      Stack% is each ticker's % move against its own previous close, so two tickers that
      trade together "should" print the same Stack%, and both legs start every session at
      exactly 0. With center_mode="zero" (recommended) the spread is measured straight from
      that natural anchor: dev = Stack%_A - beta*Stack%_B, beta fitted through the origin,
      no intercept and no re-centring. Otherwise the deviation is what they actually do
      minus what the fitted model says they should:
          hedge_mode="ols"  dev = Stack%_A - (alpha + beta * Stack%_B)
          hedge_mode="unit" dev = Stack%_A - Stack%_B - mean(Stack%_A - Stack%_B)
      alpha/beta are fitted per (pair, class) — the relationship at 03:00 is not the
      relationship at 11:00, so one global beta would smear all three classes together.
      z = dev / std(dev) within the class.

    Episode machine (per pair, per class, per session day):
      - DIVERGENCE: |z| >= div_z AND |dev| >= div_abs_pp (whichever of the two is set),
        held for >= min_hold consecutive candles.
      - PEAK: the largest |dev| that itself survived min_hold candles (a sliding minimum, so
        a one-candle spike can never set the peak).
      - CONVERGENCE: |z| <= conv_z AND |dev| <= conv_abs_pp (whichever is set), held for
        >= min_hold candles, after the divergence and inside the same day and class window.
      - A converged episode CLOSES the event. The next divergence after it opens a new one,
        so a pair can legitimately produce several episodes in one session.
      - Divergence runs that are not separated by a convergence belong to the SAME episode
        (peak = the max across them). Without this rule one unresolved divergence that
        oscillates around the threshold would be counted as a dozen separate episodes and
        inflate both TOTAL and the failure count.
      - An episode still open when the class window ends counts as a FAILURE (it is also
        exported as "unresolved" so the censored variant can be re-derived).
      - ONSET: if the class has an onset window (OPEN: 9:00-9:25), only episodes born inside
        it are rated; they may still converge anywhere up to the end of the class window.

    Per pair x class:
      total     — every divergence episode
      converged — the ones that came back
      rate      — converged / total
      rate_lb   — Wilson 95% lower bound on rate; USE THIS TO RANK, not rate. rate=1.0 out of
                  3 episodes is not better than rate=0.82 out of 200, and plain rate says it is.
      sig       — root-mean-square of the peak deviations of the CONVERGED episodes, in
                  percentage points: how far the spread stretched.
      cap_mean / cap_p50 / cap_p10 — what a trade actually BANKS: the distance from the
                  confirmed entry to the confirmed exit. You never enter at the peak, so sig
                  overstates the take; with div_abs_pp=0.5 and conv_abs_pp=0.1 the floor is
                  0.4pp. cap_p10 is the pessimistic end of the distribution.
      sig_z     — the same in z units.
      Also: separate long/short stats (dev>0 vs dev<0 — a pair is often not symmetric),
      median_bars_to_conv, corr, beta, alpha, resid_std, mr_lambda, half_life, beta_drift.

    Ranking: score = rate_lb * cap_mean — the expected REALISED take per episode, discounted
    by how confident the convergence rate actually is.
    """
    import gc, json, time, math, gzip, heapq
    from collections import defaultdict
    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq
    from pathlib import Path

    if class_windows is None:
        class_windows = CLASS_WINDOWS_DEFAULT
    if hedge_mode not in ("ols", "unit"):
        raise ValueError("hedge_mode must be 'ols' or 'unit'")
    if min_hold < 1:
        raise ValueError("min_hold must be >= 1")
    if div_z is None and div_abs_pp is None:
        raise ValueError("set at least one of div_z / div_abs_pp")
    if div_z is not None and conv_z is not None and conv_z >= div_z:
        raise ValueError(f"conv_z ({conv_z}) must be below div_z ({div_z})")
    if div_abs_pp is not None and conv_abs_pp is not None and conv_abs_pp >= div_abs_pp:
        raise ValueError(f"conv_abs_pp ({conv_abs_pp}) must be below div_abs_pp ({div_abs_pp})")
    if scale_mode not in ("std", "mad"):
        raise ValueError("scale_mode must be 'std' or 'mad'")
    if center_mode not in ("zero", "mean", "median", "auto"):
        raise ValueError("center_mode must be 'zero', 'mean', 'median' or 'auto'")
    center_median = center_mode == "median" or (
        center_mode == "auto" and (scale_mode == "mad" or
                                   div_abs_pp is not None or conv_abs_pp is not None))

    best_total_min = min_total if best_min_total is None else int(best_min_total)
    CLASSES = list(class_windows.keys())
    CLS_SMIN = {c: (_to_smin(a, session_split_min), _to_smin(b, session_split_min))
                for c, (a, b) in class_windows.items()}
    if onset_windows is None:
        onset_windows = {"OPEN": ((9, 0), (9, 25))}
    ONSET_SMIN = {}
    for c in CLASSES:
        w = onset_windows.get(c)
        ONSET_SMIN[c] = CLS_SMIN[c] if w is None else (_to_smin(w[0], session_split_min),
                                                       _to_smin(w[1], session_split_min))
        olo, ohi = ONSET_SMIN[c]
        clo, chi = CLS_SMIN[c]
        if olo < clo or ohi > chi or olo > ohi:
            raise ValueError(f"onset window for {c} ({olo}..{ohi}) must sit inside its "
                             f"class window ({clo}..{chi})")

    try:
        from statsmodels.tsa.stattools import adfuller as _adfuller
    except Exception:
        _adfuller = None

    for p in (output_onefile_jsonl, output_summary_csv, output_best_pairs_jsonl, output_episodes_jsonl):
        Path(p).parent.mkdir(parents=True, exist_ok=True)

    def _open_gz(path, mode="wt"):
        if str(path).lower().endswith(".gz"):
            return gzip.open(path, mode, encoding="utf-8", newline="\n", compresslevel=6)
        return open(path, mode.replace("t", ""), encoding="utf-8", newline="\n")

    CLS_FIELDS = ("total", "converged", "unresolved", "rate", "rate_lb", "sig", "sig_z",
                  "avg_peak", "p90_peak", "median_bars",
                  "cap_mean", "cap_p50", "cap_p10", "score",
                  "long_total", "long_rate", "long_sig",
                  "short_total", "short_rate", "short_sig",
                  "corr", "beta", "alpha", "resid_std", "mr_lambda", "half_life",
                  "beta_drift", "adf_t", "adf_p", "adf_stationary_5pct", "n_bars", "n_days")
    summary_cols = ["ticker_a", "ticker_b", "bench"] + [f"{c}_{f}" for c in CLASSES for f in CLS_FIELDS]
    pd.DataFrame(columns=summary_cols).to_csv(output_summary_csv, index=False, mode="w")

    onefile_f  = _open_gz(output_onefile_jsonl, "wt")
    episodes_f = _open_gz(output_episodes_jsonl, "wt") if write_episodes else None

    # ── small numeric helpers ────────────────────────────────────────────────
    def _js(x):
        if x is None: return None
        if isinstance(x, (np.floating, float)):
            return None if (np.isnan(x) or np.isinf(x)) else round(float(x), 6)
        if isinstance(x, (np.integer, int)): return int(x)
        if isinstance(x, (np.bool_, bool)): return bool(x)
        return x

    def _dstr(v):
        v = int(v)
        return f"{v // 10000:04d}-{(v // 100) % 100:02d}-{v % 100:02d}"

    def _wilson_lb(k, n, z=1.96):
        # Lower bound of the Wilson score interval. Shrinks small samples towards 0 instead
        # of letting 3/3 = 1.0 outrank 180/200 = 0.9.
        if n <= 0: return None
        p = k / n
        d = 1.0 + z * z / n
        c = p + z * z / (2 * n)
        m = z * math.sqrt(max(p * (1 - p) / n + z * z / (4 * n * n), 0.0))
        return max(0.0, (c - m) / d)

    def _runs(mask, brk):
        """Maximal runs of True in `mask`, additionally cut wherever brk[i] marks a
        discontinuity before position i (new session day or a hole in the candles)."""
        n = mask.size
        if n == 0:
            return np.empty(0, np.int64), np.empty(0, np.int64)
        prev = np.empty(n, bool); prev[0] = False; prev[1:] = mask[:-1]
        nxt = np.empty(n, bool); nxt[-1] = False; nxt[:-1] = mask[1:]
        brk_next = np.empty(n, bool); brk_next[-1] = True; brk_next[:-1] = brk[1:]
        starts = np.flatnonzero(mask & (~prev | brk))
        ends = np.flatnonzero(mask & (~nxt | brk_next)) + 1
        return starts, ends

    def _sustain_min(x, w):
        """y[i] = min(x[i:i+w]) — the level that held for w candles ending at i+w-1."""
        if w <= 1:
            return x
        if x.size < w:
            return np.empty(0, x.dtype)
        out = x[:x.size - w + 1].copy()
        for k in range(1, w):
            np.minimum(out, x[k:x.size - w + 1 + k], out=out)
        return out

    def _ols(x, y):
        n = x.size
        if n < 3: return 0.0, 1.0
        mx = x.mean(); my = y.mean()
        vx = float(((x - mx) ** 2).sum())
        if vx <= 0: return float(my - mx), 1.0
        beta = float(((x - mx) * (y - my)).sum() / vx)
        return float(my - beta * mx), beta

    def _mr_stats(dev, brk):
        """Day-aware mean reversion: d_dev_t = a + lam*dev_{t-1}. half_life = -ln2/ln(1+lam).
        Pairs straddling a session break are dropped, otherwise the daily reset of Stack%
        would be read as a gigantic reversion."""
        n = dev.size
        if n < 30:
            return None, None
        use = ~brk[1:]
        lag = dev[:-1][use]
        d = (dev[1:] - dev[:-1])[use]
        if lag.size < 30:
            return None, None
        a, lam = _ols(lag, d)
        # phi is the AR(1) coefficient of the spread. lam in (-1, 0) is ordinary decay.
        # lam in (-2, -1) is stationary but OSCILLATING — the spread overshoots zero every
        # bar (bid-ask bounce does exactly this on minute data) — and its envelope still
        # decays, so the half-life comes from |phi|. log1p(lam) is undefined at lam <= -1,
        # so it can never be used directly here.
        phi = 1.0 + lam
        if lam >= 0 or abs(phi) >= 1.0:
            return _js(lam), None
        if phi == 0.0:
            return _js(lam), 0.0          # full reversion inside one bar
        hl = -math.log(2.0) / math.log(abs(phi))
        return _js(lam), _js(hl)

    # Large-sample Dickey-Fuller critical values, constant / no trend.
    ADF_CRIT = {"10%": -2.57, "5%": -2.86, "1%": -3.43}

    def _adf(dev, brk):
        """-> (t_stat, p_value, stationary_at_5pct).

        p_value is only filled when statsmodels is importable — deriving a MacKinnon p-value
        by hand would mean hard-coding response-surface coefficients, and a wrong p-value is
        worse than none. Without statsmodels you still get the t-stat and the verdict against
        the standard critical value (5% = -2.86), which is what the decision actually needs.
        `pip install statsmodels` if you want the exact p."""
        if not compute_adf or dev.size < 50:
            return None, None, None
        if _adfuller is not None:
            try:
                r = _adfuller(dev, maxlag=adf_maxlag, autolag=None)
                return _js(r[0]), _js(r[1]), bool(r[1] < 0.05)
            except Exception:
                return None, None, None
        # numpy fallback: plain Dickey-Fuller with a constant (no augmentation), day-aware
        use = ~brk[1:]
        lag = dev[:-1][use]
        d = (dev[1:] - dev[:-1])[use]
        if lag.size < 50:
            return None, None, None
        X = np.column_stack([np.ones(lag.size), lag])
        coef, res, *_ = np.linalg.lstsq(X, d, rcond=None)
        resid = d - X @ coef
        dof = lag.size - 2
        if dof <= 0:
            return None, None, None
        s2 = float(resid @ resid) / dof
        xtx_inv = np.linalg.inv(X.T @ X)
        se = math.sqrt(max(s2 * xtx_inv[1, 1], 1e-30))
        t = float(coef[1] / se)
        return _js(t), None, bool(t < ADF_CRIT["5%"])

    def _pairwise_corr(R):
        """Masked pairwise correlation of every column against every other, tolerating NaN
        holes without dropping whole rows. Five matmuls instead of N^2 python pairs."""
        W = np.isfinite(R).astype(np.float32)
        X = np.where(np.isfinite(R), R, 0.0).astype(np.float32)
        X2 = X * X
        n = W.T @ W
        sx = X.T @ W
        sy = W.T @ X
        sxx = X2.T @ W
        syy = W.T @ X2
        sxy = X.T @ X
        with np.errstate(invalid="ignore", divide="ignore"):
            cov = n * sxy - sx * sy
            vx = n * sxx - sx * sx
            vy = n * syy - sy * sy
            c = cov / np.sqrt(vx * vy)
        c[~np.isfinite(c)] = np.nan
        np.fill_diagonal(c, np.nan)
        c[n < 30] = np.nan
        return c

    def _episodes(dev, z, sdate, brk):
        """-> (ep_start_idx, peak, converged, bars_to_conv, direction) as numpy arrays."""
        absz = np.abs(z)
        absd = np.abs(dev)
        dmask = (absz >= div_z) if div_z is not None else np.ones(absd.size, bool)
        if div_abs_pp is not None:
            dmask = dmask & (absd >= div_abs_pp)
        ds, de = _runs(dmask, brk)
        keep = (de - ds) >= min_hold
        ds, de = ds[keep], de[keep]
        if ds.size == 0:
            return None
        cmask = (absz <= conv_z) if conv_z is not None else np.ones(absd.size, bool)
        if conv_abs_pp is not None:
            cmask = cmask & (absd <= conv_abs_pp)
        cs, ce = _runs(cmask, brk)
        cs = cs[(ce - cs) >= min_hold]

        sm = _sustain_min(np.abs(dev), min_hold)
        if sm.size == 0:
            return None
        idx = np.empty(2 * ds.size, dtype=np.int64)
        idx[0::2] = np.minimum(ds, sm.size - 1)
        idx[1::2] = np.clip(de - min_hold + 1, 0, sm.size - 1)
        run_peak = np.maximum.reduceat(sm, idx)[0::2]

        # the convergence run that resolves each divergence run; equal values == same episode
        res = np.searchsorted(cs, de)
        sd = sdate[ds]
        new = np.empty(ds.size, bool); new[0] = True
        new[1:] = (res[1:] != res[:-1]) | (sd[1:] != sd[:-1])
        g = np.flatnonzero(new)

        ep_start = ds[g]
        ep_peak = np.maximum.reduceat(run_peak, g)
        ep_res = res[g]
        ok = ep_res < cs.size
        conv_pos = np.where(ok, cs[np.clip(ep_res, 0, max(cs.size - 1, 0))] if cs.size else 0, -1)
        converged = ok & (conv_pos >= 0)
        if cs.size:
            converged &= sdate[np.clip(conv_pos, 0, sdate.size - 1)] == sdate[ep_start]
        bars = np.where(converged, conv_pos - ep_start, -1)
        direction = np.sign(dev[ep_start])
        # What the trade actually banks. You do not enter at the peak: you enter when the
        # divergence is CONFIRMED (min_hold candles past the threshold) and you leave when
        # the convergence is confirmed. capture is the distance travelled between those two
        # points, signed so that moving toward zero is positive — an overshoot past zero
        # counts as extra. peak/sig describe how far the spread stretched; capture is the
        # only number that answers "how much do I take home".
        n_dev = dev.size
        entry_dev = dev[np.minimum(ep_start + min_hold - 1, n_dev - 1)]
        exit_dev = np.where(converged, dev[np.clip(conv_pos + min_hold - 1, 0, n_dev - 1)], np.nan)
        capture = np.where(converged, np.sign(entry_dev) * (entry_dev - exit_dev), np.nan)
        return ep_start, ep_peak, converged, bars, direction, entry_dev, capture

    # ── per-benchmark scan ───────────────────────────────────────────────────
    stage = Path(stage_dir)
    files = sorted(stage.glob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"no staged parquet files in {stage_dir} — run stage 1 first")

    best_heaps = {c: [] for c in CLASSES}
    t0 = time.time()
    pairs_written = 0

    print(f"START PairFlux stage2  benches={len(files)}  hedge={hedge_mode}  "
          f"div_z={div_z} conv_z={conv_z} min_hold={min_hold}  min_corr={min_corr}")

    for fp in files:
        bench = fp.stem
        tb0 = time.time()
        df = pq.read_table(fp).to_pandas()
        if df.empty:
            continue

        tickers, tk_code = np.unique(df["ticker"].to_numpy(), return_inverse=True)
        # coverage guard before anything expensive
        cov = np.bincount(tk_code, minlength=tickers.size)
        ndays = pd.Series(df["sdate"].to_numpy()).groupby(tk_code).nunique().reindex(
            range(tickers.size)).fillna(0).to_numpy()
        good = (cov >= min_bars_per_ticker) & (ndays >= min_days_per_ticker)
        n_cov = int(good.sum())
        if n_cov < tickers.size:
            print(f"  [{bench}] {tickers.size - n_cov} of {tickers.size} tickers dropped by "
                  f"coverage (min_bars={min_bars_per_ticker}, min_days={min_days_per_ticker})")
        if n_cov > max_tickers_per_bench:
            # rank WITHIN the eligible set, and say so — this is a real narrowing of
            # "check every ticker" and must never happen silently
            elig = np.flatnonzero(good)
            keep = elig[np.argsort(-cov[elig])[:max_tickers_per_bench]]
            good[:] = False
            good[keep] = True
            print(f"  [{bench}] CAPPED to the {max_tickers_per_bench} best-covered tickers of "
                  f"{n_cov} eligible — raise max_tickers_per_bench to widen the scan")
        if good.sum() < 2:
            print(f"  [{bench}] skipped — only {int(good.sum())} tickers pass coverage")
            continue

        sel = np.flatnonzero(good)
        remap = -np.ones(tickers.size, np.int64)
        remap[sel] = np.arange(sel.size)
        keep_rows = remap[tk_code] >= 0
        col_of_row = remap[tk_code[keep_rows]]
        sdate_all = df["sdate"].to_numpy()[keep_rows]
        smin_all = df["smin"].to_numpy().astype(np.int32)[keep_rows]
        stack_all = df["stack"].to_numpy()[keep_rows]
        names = tickers[sel]
        del df
        gc.collect()

        if bar_minutes is None:
            s = np.sort(np.unique(smin_all))
            d = np.diff(s)
            d = d[d > 0]
            step = int(np.bincount(d).argmax()) if d.size else 1
        else:
            step = int(bar_minutes)
        gap_tol = step if max_gap_minutes is None else max(step, int(max_gap_minutes))

        pair_stats = defaultdict(dict)

        for cls in CLASSES:
            lo, hi = CLS_SMIN[cls]
            m = (smin_all >= lo) & (smin_all <= hi)
            if m.sum() < min_bars_per_ticker:
                continue
            sd_c = sdate_all[m]; sm_c = smin_all[m]
            col_c = col_of_row[m]; val_c = stack_all[m]

            row_key = sd_c.astype(np.int64) * 100000 + (sm_c.astype(np.int64) + 1440)
            uniq_rows, row_idx = np.unique(row_key, return_inverse=True)
            T, N = uniq_rows.size, names.size
            mb = T * N * 4 / 1e6
            if mb > max_matrix_mb:
                print(f"  [{bench}/{cls}] SKIPPED — matrix would be {mb:,.0f} MB "
                      f"({T:,} rows x {N} tickers). Narrow start_date or max_tickers_per_bench.")
                continue

            M = np.full((T, N), np.nan, dtype=np.float32)
            M[row_idx, col_c] = val_c
            r_sdate = (uniq_rows // 100000).astype(np.int32)
            r_smin = (uniq_rows % 100000 - 1440).astype(np.int32)
            brk = np.empty(T, bool); brk[0] = True
            brk[1:] = (r_sdate[1:] != r_sdate[:-1]) | (r_smin[1:] - r_smin[:-1] > gap_tol)

            # candidate filter on RETURNS, not on Stack% levels: two tickers both drifting up
            # all session correlate ~1 on levels no matter how they got there.
            kbar = max(1, int(corr_step_bars))
            if T <= kbar:
                del M
                gc.collect()
                continue
            cbrk = np.cumsum(brk.astype(np.int32))
            R = M[kbar:] - M[:-kbar]
            # a k-bar return is only valid if no session break or candle gap falls inside it
            R[(cbrk[kbar:] - cbrk[:-kbar]) > 0] = np.nan
            Rc = R
            if R.shape[0] > corr_max_rows:
                Rc = R[np.linspace(0, R.shape[0] - 1, corr_max_rows).astype(np.int64)]
            C = _pairwise_corr(Rc)
            iu = np.triu_indices(N, k=1)
            cvals = C[iu]
            cand = np.flatnonzero(np.isfinite(cvals) & (cvals >= min_corr))
            if cand.size == 0:
                print(f"  [{bench}/{cls}] no pair reaches corr>={min_corr} "
                      f"(best={np.nanmax(cvals) if np.isfinite(cvals).any() else float('nan'):.3f})")
                del M, R, C
                gc.collect()
                continue
            if cand.size > max_pairs_per_bench:
                cand = cand[np.argsort(-cvals[cand])[:max_pairs_per_bench]]
            ai, bi = iu[0][cand], iu[1][cand]
            print(f"  [{bench}/{cls}] rows={T:,} tickers={N} pairs={cand.size:,} "
                  f"({mb:,.0f} MB matrix, step={step}m)")

            for k in range(cand.size):
                ia, ib = int(ai[k]), int(bi[k])
                a = M[:, ia]; b = M[:, ib]
                v = np.isfinite(a) & np.isfinite(b)
                if v.sum() < min_bars_per_ticker:
                    continue
                va = a[v].astype(np.float64); vb = b[v].astype(np.float64)
                sdv = r_sdate[v]
                # recompute breaks on the pair's own valid grid: a hole in EITHER leg breaks
                # the run, otherwise "3 consecutive candles" would silently span a gap
                smv = r_smin[v]
                bv = np.empty(va.size, bool); bv[0] = True
                bv[1:] = (sdv[1:] != sdv[:-1]) | (smv[1:] - smv[:-1] > gap_tol)

                if center_mode == "zero":
                    # Stack% is each ticker's move against its OWN previous close, so both
                    # legs start every session at exactly 0. The spread therefore has a real
                    # anchor at zero and must not be re-centred: beta is fitted THROUGH THE
                    # ORIGIN and alpha is pinned to 0, making dev literally A - beta*B. A
                    # fitted intercept would move "no deviation" off true parity, and a
                    # persistent one-sided drift would then be silently absorbed into it.
                    if hedge_mode == "ols":
                        den = float(vb @ vb)
                        beta = float((va @ vb) / den) if den > 0 else 1.0
                    else:
                        beta = 1.0
                    alpha = 0.0
                elif hedge_mode == "ols":
                    alpha, beta = _ols(vb, va)
                else:
                    # beta pinned to 1, but alpha still centres the spread so that "dev == 0"
                    # means the same thing in both modes: the pair sits at its own equilibrium
                    beta = 1.0
                    alpha = float((va - vb).mean())
                if beta_band is not None and not (1.0 / beta_band <= beta <= beta_band):
                    continue
                dev = va - (alpha + beta * vb)
                if center_median:
                    med = float(np.median(dev))
                    dev = dev - med
                    alpha += med
                if scale_mode == "mad":
                    sc = float(np.median(np.abs(dev - np.median(dev)))) * 1.4826
                    # MAD collapses to 0 on a spread that is flat more than half the time
                    sd_dev = sc if sc > 1e-9 else float(dev.std())
                else:
                    sd_dev = float(dev.std())
                if not np.isfinite(sd_dev) or sd_dev <= 1e-9:
                    continue
                z = dev / sd_dev

                ep = _episodes(dev, z, sdv, bv)
                if ep is None:
                    continue
                ep_start, peak, conv, bars, dirn, entry_dev, capture = ep
                olo, ohi = ONSET_SMIN[cls]
                if (olo, ohi) != (lo, hi) or require_fresh_onset:
                    m = (smv[ep_start] >= olo) & (smv[ep_start] <= ohi)
                    if require_fresh_onset:
                        # a run beginning exactly on a discontinuity (day start or a hole in
                        # the candles) has an unknown birth time — it is not an onset
                        m &= ~bv[ep_start]
                    if not m.any():
                        continue
                    ep_start, peak, conv, bars, dirn, entry_dev, capture = (
                        ep_start[m], peak[m], conv[m], bars[m], dirn[m],
                        entry_dev[m], capture[m])
                if min_abs_peak_pp > 0 or max_abs_peak_pp > 0:
                    m = np.ones(peak.size, bool)
                    if min_abs_peak_pp > 0:
                        m &= peak >= min_abs_peak_pp
                    if max_abs_peak_pp > 0:
                        m &= peak <= max_abs_peak_pp
                    if not m.any():
                        continue
                    ep_start, peak, conv, bars, dirn, entry_dev, capture = (
                        ep_start[m], peak[m], conv[m], bars[m], dirn[m],
                        entry_dev[m], capture[m])
                total = int(ep_start.size)
                nconv = int(conv.sum())
                pk_c = peak[conv]
                sig = float(np.sqrt((pk_c ** 2).mean())) if pk_c.size else None
                rate = nconv / total if total else None
                rate_lb = _wilson_lb(nconv, total)
                cap_c = capture[conv]
                cap_c = cap_c[np.isfinite(cap_c)]
                cap_mean = float(cap_c.mean()) if cap_c.size else None
                cap_p50 = float(np.median(cap_c)) if cap_c.size else None
                # the pessimistic end: 1 converged episode in 10 gives you no more than this
                cap_p10 = float(np.percentile(cap_c, 10)) if cap_c.size else None

                def _dir_stats(sign):
                    dm = dirn == sign
                    tt = int(dm.sum())
                    if tt == 0: return 0, None, None
                    cc = conv & dm
                    pk = peak[cc]
                    return (tt, round(int(cc.sum()) / tt, 4),
                            _js(float(np.sqrt((pk ** 2).mean())) if pk.size else None))

                lt, lr, ls = _dir_stats(1.0)
                st_, sr, ss = _dir_stats(-1.0)

                lam, hl = _mr_stats(dev, bv)
                adf_t, adf_p, adf_s5 = _adf(dev, bv)
                # split-half beta: a pair whose hedge ratio drifts is not the same pair any more
                half = va.size // 2
                if hedge_mode == "ols" and half > 30:
                    _, b1 = _ols(vb[:half], va[:half])
                    _, b2 = _ols(vb[half:], va[half:])
                    bdrift = abs(b2 - b1)
                else:
                    bdrift = None

                key = (str(names[ia]), str(names[ib]))
                pair_stats[key][cls] = {
                    "total": total, "converged": nconv, "unresolved": total - nconv,
                    "rate": _js(rate), "rate_lb": _js(rate_lb),
                    "sig": _js(sig), "sig_z": _js(sig / sd_dev if sig is not None else None),
                    "avg_peak": _js(float(pk_c.mean()) if pk_c.size else None),
                    "p90_peak": _js(float(np.percentile(pk_c, 90)) if pk_c.size else None),
                    "median_bars": _js(float(np.median(bars[conv])) if nconv else None),
                    "cap_mean": _js(cap_mean), "cap_p50": _js(cap_p50), "cap_p10": _js(cap_p10),
                    # ranked on REALISED capture, not on the peak: rate_lb * cap_mean is the
                    # confidence-discounted expected take per converged episode
                    "score": _js((rate_lb or 0.0) * (cap_mean or 0.0)),
                    "long_total": lt, "long_rate": lr, "long_sig": ls,
                    "short_total": st_, "short_rate": sr, "short_sig": ss,
                    "corr": _js(float(cvals[cand[k]])),
                    "beta": _js(beta), "alpha": _js(alpha), "resid_std": _js(sd_dev),
                    "mr_lambda": lam, "half_life": hl, "beta_drift": _js(bdrift),
                    "adf_t": adf_t, "adf_p": adf_p, "adf_stationary_5pct": adf_s5,
                    "n_bars": int(va.size), "n_days": int(np.unique(sdv).size),
                }

                if write_episodes:
                    a_n, b_n = key
                    for j in range(total):
                        episodes_f.write(json.dumps({
                            "a": a_n, "b": b_n, "bench": bench, "cls": cls,
                            "date": _dstr(sdv[ep_start[j]]),
                            "peak": _js(float(peak[j])),
                            "peak_z": _js(float(peak[j] / sd_dev)),
                            "entry_dev": _js(float(entry_dev[j])),
                            "capture": _js(float(capture[j])) if np.isfinite(capture[j]) else None,
                            "converged": bool(conv[j]),
                            "bars": int(bars[j]),
                            "dir": int(dirn[j]),
                        }, ensure_ascii=False) + "\n")

                if log_every_n_pairs and (k + 1) % log_every_n_pairs == 0:
                    print(f"    ...{k+1:,}/{cand.size:,} pairs  elapsed={time.time()-tb0:.1f}s")

            del M, R, C
            gc.collect()

        # ── emit this benchmark's pairs ──
        rows = []
        for (a_n, b_n), per_cls in pair_stats.items():
            if not any((per_cls.get(c) or {}).get("total", 0) >= min_total for c in CLASSES):
                continue
            onefile_f.write(json.dumps({
                "a": a_n, "b": b_n, "bench": bench,
                "params": {
                    "hedge_mode": hedge_mode, "div_z": div_z, "conv_z": conv_z,
                    "min_hold": min_hold, "min_corr": min_corr,
                    "class_windows": {c: [list(x) for x in class_windows[c]] for c in CLASSES},
                    "onset_smin": {c: list(ONSET_SMIN[c]) for c in CLASSES},
                    "require_fresh_onset": require_fresh_onset,
                    "div_z": div_z, "conv_z": conv_z,
                    "scale_mode": scale_mode, "center_median": center_median,
                    "div_abs_pp": div_abs_pp, "conv_abs_pp": conv_abs_pp,
                    "max_gap_minutes": max_gap_minutes, "gap_tol": gap_tol,
                    "min_abs_peak_pp": min_abs_peak_pp, "max_abs_peak_pp": max_abs_peak_pp,
                    "session_split_min": session_split_min, "bar_minutes": step,
                },
                "classes": per_cls,
            }, ensure_ascii=False) + "\n")
            row = {"ticker_a": a_n, "ticker_b": b_n, "bench": bench}
            for c in CLASSES:
                d = per_cls.get(c) or {}
                for f in CLS_FIELDS:
                    row[f"{c}_{f}"] = d.get(f)
                if (d.get("converged", 0) > 0 and d.get("total", 0) >= best_total_min
                        and d.get("score")):
                    h = best_heaps[c]
                    item = (d["score"], a_n, b_n, bench, d.get("rate"), d.get("rate_lb"),
                            d.get("sig"), d.get("total"))
                    if len(h) < top_k_best:
                        heapq.heappush(h, item)
                    elif item[0] > h[0][0]:
                        heapq.heapreplace(h, item)
            rows.append(row)
            pairs_written += 1

        if rows:
            pd.DataFrame(rows, columns=summary_cols).to_csv(
                output_summary_csv, mode="a", header=False, index=False)
        print(f"  [{bench}] pairs kept={len(rows):,}  elapsed={time.time()-tb0:.1f}s")
        del pair_stats
        gc.collect()

    with _open_gz(output_best_pairs_jsonl, "wt") as bf:
        from datetime import datetime as _dtm
        bf.write(json.dumps({"meta": {
            "version": "pairflux_v1",
            "generated_at": _dtm.utcnow().isoformat() + "Z",
            "ranked_by": "score = rate_lb * sig",
        }}) + "\n")
        for c in CLASSES:
            top = sorted(best_heaps[c], key=lambda x: -x[0])
            bf.write(json.dumps({"cls": c, "top": [
                {"a": a, "b": b, "bench": bn, "score": _js(s), "rate": _js(r),
                 "rate_lb": _js(rl), "sig": _js(sg), "total": t}
                for (s, a, b, bn, r, rl, sg, t) in top
            ]}, ensure_ascii=False) + "\n")

    onefile_f.close()
    if episodes_f is not None:
        episodes_f.close()
    print(f"DONE PairFlux pairs={pairs_written:,} elapsed={time.time()-t0:.1f}s")
    print(f"  onefile    = {output_onefile_jsonl}")
    print(f"  summary    = {output_summary_csv}")
    print(f"  best_pairs = {output_best_pairs_jsonl}")
    print(f"  episodes   = {output_episodes_jsonl if write_episodes else '(disabled)'}")

In [5]:
from pathlib import Path
import os


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env   = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path   = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve()   if sig_env   else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break

        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var (recommended).")

        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")

    return final_path, out_dir


# ── runner ────────────────────────────────────────────────────────────────────

FINAL_PATH, OUT_DIR = _resolve_orion_paths("pairflux")
STAGE_DIR = OUT_DIR / "_stage"

# Stage 1 is the slow part and only depends on the class windows / date range. Once it has
# run you can iterate on thresholds by re-running stage 2 alone.
RUN_STAGE1 = True

if RUN_STAGE1:
    pairflux_stage1_shuffle(
        input_path=str(FINAL_PATH),
        stage_dir=str(STAGE_DIR),
        class_windows=CLASS_WINDOWS_DEFAULT,
        session_split_min=1020,     # 17:00 — everything later belongs to the next session
        start_date=None,            # e.g. "2026-01-01" to cut history and memory
        bench_whitelist=None,       # e.g. ["SPY", "IWM"] to test on two groups first
        STOCK_NUM_FIELD="Stack%",
    )

pairflux_stats_exporter(
    stage_dir=str(STAGE_DIR),
    output_onefile_jsonl=str(OUT_DIR / "onefile.jsonl.gz"),
    output_summary_csv=str(OUT_DIR / "summary.csv"),
    output_best_pairs_jsonl=str(OUT_DIR / "best_pairs.jsonl.gz"),
    output_episodes_jsonl=str(OUT_DIR / "episodes.jsonl.gz"),
    write_episodes=True,
    class_windows=CLASS_WINDOWS_DEFAULT,
    # OPEN rates only the deviations BORN in 9:00-9:25, but still gives them until 10:00
    # to normalise. PRE/INTRA count onsets anywhere inside their own window.
    onset_windows={"OPEN": ((9, 0), (9, 25))},
    require_fresh_onset=True,
    session_split_min=1020,
    bar_minutes=None,               # infer from data
    hedge_mode="ols",               # "unit" = plain Stack%_A - Stack%_B
    # Divergence strength is now measured in PERCENTAGE POINTS, as specified: 0.5pp opens
    # an event, back inside 0.1pp closes it. div_z/conv_z=None turns the sigma test off
    # entirely — if this floods you with episodes from pairs whose ordinary noise is already
    # ~0.5pp wide, put div_z=1.5 back to require the move be unusual for THAT pair too.
    div_z=None, div_abs_pp=0.5,
    conv_z=None, conv_abs_pp=0.1,
    min_hold=3,
    scale_mode="std",               # switch to "mad" if a class comes back empty
    # zero = the pair's TYPICAL state (median-centred), so a divergence is measured from
    # where the pair normally sits. "zero" instead measures from literal parity A - beta*B.
    center_mode="auto",
    # measured on real data: overnight/pre-market bars are 2-4 min apart, so a strict
    # 1-minute adjacency rule prevents PRE/OPEN episodes from ever forming
    max_gap_minutes=5,
    min_abs_peak_pp=0.0,            # e.g. 0.3 to ignore untradeably small divergences
    max_abs_peak_pp=0.0,            # e.g. 15.0 to drop news-driven pseudo-divergences
    min_corr=0.7, corr_step_bars=5,
    max_pairs_per_bench=20000,
    min_bars_per_ticker=500, min_days_per_ticker=10,
    max_tickers_per_bench=800, max_matrix_mb=2000,
    min_total=5,
    best_min_total=10,              # the ranked list needs more evidence than the CSV does
    beta_band=None,                 # 1.5 keeps only genuinely 1:1 pairs (drops geared ETFs)
    top_k_best=500,
    compute_adf=False,              # see the note in the docstring before turning this on
)


START PairFlux stage1  file=C:\datum-api-examples-main\OriON\CRACEN\final.parquet
  session_split=1020min  smin window=[-180, 960]  start_date=None


[rg   20/7803] in=471,582 staged=415,233 benches=11 elapsed=0.6s


[rg   40/7803] in=972,769 staged=869,734 benches=17 elapsed=1.1s


[rg   60/7803] in=1,377,136 staged=1,247,851 benches=17 elapsed=1.6s


[rg  100/7803] in=2,044,402 staged=1,884,861 benches=19 elapsed=2.9s


...

  [XOP/PRE] rows=36,536 tickers=135 pairs=1 (20 MB matrix, step=1m)
  [XOP/OPEN] rows=3,729 tickers=135 pairs=36 (2 MB matrix, step=1m)


  [XOP/INTRA] rows=21,807 tickers=135 pairs=43 (12 MB matrix, step=1m)
  [XOP] pairs kept=41  elapsed=2.2s


  [XRT] 5 of 169 tickers dropped by coverage (min_bars=500, min_days=10)


  [XRT/PRE] rows=39,038 tickers=164 pairs=2 (26 MB matrix, step=1m)
  [XRT/OPEN] rows=3,780 tickers=164 pairs=4 (2 MB matrix, step=1m)


  [XRT/INTRA] rows=22,538 tickers=164 pairs=2 (15 MB matrix, step=1m)
  [XRT] pairs kept=1  elapsed=2.1s
DONE PairFlux pairs=16,735 elapsed=154.6s
  onefile    = C:\datum-api-examples-main\OriON\signals\pairflux\onefile.jsonl.gz
  summary    = C:\datum-api-examples-main\OriON\signals\pairflux\summary.csv
  best_pairs = C:\datum-api-examples-main\OriON\signals\pairflux\best_pairs.jsonl.gz
  episodes   = C:\datum-api-examples-main\OriON\signals\pairflux\episodes.jsonl.gz
